# TradeSense — Layer 1 Final Artifact Audit (BTC/ETH)

**Purpose:** Safely verify the **already completed** 90-day Layer 1 outputs before model training or committing to GitHub. This notebook is **read-only**: it does **not** download Binance archives, regenerate features, overwrite Parquet files, or fit scalers. The original production notebook remains the **source-code provenance** for raw-data rebuilding; this notebook is **not a replacement for a full reproducible build pipeline**.

Expected: 2024-05-01 through 2024-07-29 UTC, six features, 129,600 rows per symbol. The initial May 1 rolling-feature warm-up must be masked for model training unless previous-day context was supplied.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Colab unavailable: set BASE to your local mounted dataset path.')
BASE = Path('/content/drive/MyDrive/research datasets')
assert BASE.exists(), f'Dataset root not mounted: {BASE}'


In [ ]:
ROOT = BASE / 'processed_layer1_final_v1'
FEATURES = ['log_returns','trade_flow_imbalance','hl_spread_ratio','rush_order_density','volume_acceleration','large_trade_count']
RANGES = {
    'train': ('2024-05-01','2024-07-02',90720),
    'val': ('2024-07-03','2024-07-15',18720),
    'test': ('2024-07-16','2024-07-29',20160),
}
assert ROOT.exists(), f'Layer 1 folder missing: {ROOT}'
print('Read-only audit:', ROOT)


In [ ]:
matrices = {}
for symbol in ('BTCUSDT','ETHUSDT'):
    path = ROOT / 'processed_matrices' / f'{symbol.lower()}_l1_feature_matrix_90d.parquet'
    df = pd.read_parquet(path)
    assert 'datetime' in df.columns
    df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
    df = df.sort_values('datetime').reset_index(drop=True)
    expected = pd.date_range('2024-05-01', periods=129600, freq='min', tz='UTC')
    assert len(df) == 129600, (symbol, len(df))
    assert df['datetime'].equals(pd.Series(expected, name='datetime')), f'{symbol}: timestamp grid mismatch'
    assert set(FEATURES).issubset(df.columns), f'{symbol}: missing features'
    assert not df[FEATURES].isna().any().any(), f'{symbol}: NaNs'
    assert np.isfinite(df[FEATURES].to_numpy(dtype=float)).all(), f'{symbol}: infs'
    assert df['trade_flow_imbalance'].between(-1.00001,1.00001).all()
    assert (df['large_trade_count'] >= 0).all()
    assert (df['hl_spread_ratio'] >= 0).all()
    matrices[symbol] = df
    print(f'PASS {symbol}: {len(df):,} rows, {len(df.columns)} columns, full UTC grid, finite features')


In [ ]:
# Dedicated partitions: compare their data against the full matrix (not only row counts).
for symbol, df in matrices.items():
    for split, (start, end, expected_count) in RANGES.items():
        p = ROOT / 'partitions' / f'{symbol.lower()}_{split}_split_{"70" if split=="train" else "15"}.parquet'
        assert p.exists(), f'Missing dedicated partition: {p}'
        part = pd.read_parquet(p)
        part['datetime'] = pd.to_datetime(part['datetime'], utc=True)
        expected = df.loc[df['datetime'].between(pd.Timestamp(start, tz='UTC'),pd.Timestamp(end, tz='UTC') + pd.Timedelta(days=1)-pd.Timedelta(minutes=1))].reset_index(drop=True)
        assert len(part) == expected_count == len(expected), (symbol,split,len(part))
        assert part['datetime'].reset_index(drop=True).equals(expected['datetime']), (symbol,split,'timestamps')
        assert np.allclose(part[FEATURES].to_numpy(dtype=float),expected[FEATURES].to_numpy(dtype=float),rtol=1e-6,atol=1e-8), (symbol,split,'features differ')
        print(f'PASS {symbol} {split}: {len(part):,} rows, values match full matrix')


In [ ]:
# Causal warm-up caution: first 15 minutes of May 1 have no previous-day context
# unless Apr 30 was explicitly loaded. Do not treat synthetic padding as genuine history.
for symbol, df in matrices.items():
    print('\n', symbol, 'initial rows (inspect before training):')
    display(df[['datetime','volume','log_returns','volume_acceleration']].head(17))
    print('Volume acceleration quantiles:')
    display(df['volume_acceleration'].quantile([0,.01,.05,.5,.95,.99,1]).to_frame('value'))
print('Recommendation: exclude initial 15 warm-up minutes from training/validation scoring if no Apr 30 context exists.')


In [ ]:
# Read metadata without changing it. Calibration was 1% systematic subsampling,
# NOT an exact all-trades percentile and NOT a true random reservoir sample.
import json
for filename in ['thresholds_calibrated.json','volume_acceleration_stabilization_config.json']:
    p = ROOT / 'metadata' / filename
    print('\n',filename)
    if p.exists():
        print(json.dumps(json.loads(p.read_text(encoding='utf-8')),indent=2)[:3000])
    else: print('NOT FOUND:',p)
print('\nREAD-ONLY AUDIT COMPLETED. Does not prove source-to-feature formula correctness.')
